# LeetCode 3329 — Count Substrings With K-Frequency Characters II bằng Python

- Link: https://leetcode.com/problems/count-substrings-with-k-frequency-characters-ii/
- Độ khó: **Hard** · **Premium**
- Chủ đề: **Cửa sổ trượt / hai con trỏ** với tính chất **đơn điệu theo đầu trái**, đếm theo đầu phải (`ans += l`), cách nhìn **phần bù** và **vị trí lần xuất hiện thứ k từ cuối**

> **Ghi chú Premium:** bài Premium nên LeetCode không trả đề qua API công khai. Đề, ví dụ, ràng buộc và chữ ký `numberOfSubstrings(self, s, k)` lấy theo trang doocs (https://leetcode.doocs.org/en/lc/3329/) — vui lòng **đối chiếu lại** nếu có tài khoản Premium.

Skeleton (theo doocs):

```python
class Solution:
    def numberOfSubstrings(self, s: str, k: int) -> int:
```

## 1. Mục tiêu học

1. Nhận ra tính **đơn điệu**: nếu `s[l..r]` có một ký tự xuất hiện `≥ k` lần thì mọi `s[l'..r]` với `l' ≤ l` cũng vậy.
2. Với mỗi đầu phải `r`, gọi `l(r)` = số đầu trái hợp lệ — chúng là **tiền tố** `0..l(r) − 1` ⇒ đáp án `= Σ_r l(r)`.
3. Duy trì `l(r)` bằng cửa sổ trượt: cửa sổ `[l, r]` là đoạn **dài nhất** kết thúc tại `r` mà mọi ký tự xuất hiện `< k` lần.
4. Cách nhìn thứ hai: `l(r) = max_c (vị trí lần xuất hiện thứ k từ cuối của c trong s[0..r]) + 1`.
5. Kiểm chứng bằng vét cạn.

## 2. Đề bài & ví dụ

Cho chuỗi `s` và số nguyên `k`. Đếm số **xâu con** (theo vị trí) của `s` mà **ít nhất một ký tự** xuất hiện **ít nhất `k` lần**.

**Ví dụ 1:** `s = "abacb"`, `k = 2` → `4` (`"aba"`, `"abac"`, `"abacb"`, `"bacb"`).
**Ví dụ 2:** `s = "abcde"`, `k = 1` → `15` (mọi xâu con).

**Ràng buộc (theo doocs):** `1 ≤ len(s) ≤ 3·10^5`; `1 ≤ k ≤ len(s)`; chữ thường.

Bản I (LC 3325) giống hệt nhưng `len(s) ≤ 3000` — cho phép `O(n²)`.

In [1]:
from typing import List
from collections import Counter

examples = [("abacb", 2, 4), ("abcde", 1, 15)]
for s, k, exp in examples:
    print(repr(s), 'k =', k, '-> kỳ vọng', exp)

'abacb' k = 2 -> kỳ vọng 4
'abcde' k = 1 -> kỳ vọng 15


## 3. Kiến thức nền

### 3.1. Đếm xâu con theo đầu phải

Mọi xâu con được xác định duy nhất bởi `(l, r)`. Gom theo `r`: đáp án `= Σ_r #{l : s[l..r] hợp lệ}`.

### 3.2. Đơn điệu theo đầu trái

Mở rộng xâu con sang trái chỉ **tăng** số lần xuất hiện ⇒ tập `l` hợp lệ (với `r` cố định) là `{0, 1, …, L − 1}` với một ngưỡng `L = l(r)`.

### 3.3. Ngưỡng `l(r)` không giảm theo `r`

Nếu `s[l..r]` hợp lệ thì `s[l..r+1]` cũng hợp lệ ⇒ `l(r + 1) ≥ l(r)`. Con trỏ trái chỉ tiến ⇒ tổng `O(n)`.

In [2]:
def l_of_r_brute(s, k):
    n = len(s)
    out = []
    for r in range(n):
        L = 0
        for l in range(r + 1):
            if max(Counter(s[l:r + 1]).values()) >= k:
                L = l + 1
        out.append(L)
    return out

print('l(r) của "abacb", k=2:', l_of_r_brute("abacb", 2), '-> tổng', sum(l_of_r_brute("abacb", 2)))

l(r) của "abacb", k=2: [0, 0, 1, 1, 2] -> tổng 4


### 3.4. Kiểm tra thực nghiệm hai tính chất đơn điệu

Trước khi tin vào hai con trỏ, ta kiểm tra bằng máy trên nhiều chuỗi ngẫu nhiên: (a) với `r` cố định, tập `l` hợp lệ đúng là một **tiền tố** `0..l(r) − 1`; (b) dãy `l(r)` **không giảm**. Đây là thói quen tốt trước khi viết cửa sổ trượt — nếu một trong hai tính chất sai, hai con trỏ sẽ cho kết quả sai một cách âm thầm.

In [3]:
import random as _r
_g = _r.Random(7)
for _ in range(300):
    s_ = ''.join(_g.choice('abc') for _ in range(_g.randint(1, 9)))
    k_ = _g.randint(1, 3)
    n_ = len(s_)
    Ls = []
    for r_ in range(n_):
        ok = [max(Counter(s_[l_:r_ + 1]).values()) >= k_ for l_ in range(r_ + 1)]
        L_ = sum(ok)
        assert ok == [True] * L_ + [False] * (r_ + 1 - L_)      # (a) tiền tố
        Ls.append(L_)
    assert all(a <= b for a, b in zip(Ls, Ls[1:]))               # (b) không giảm
print('Hai tính chất đơn điệu đúng trên 300 chuỗi ngẫu nhiên.')

Hai tính chất đơn điệu đúng trên 300 chuỗi ngẫu nhiên.


### 3.5. Vì sao chỉ cần kiểm tra ký tự vừa thêm

Cửa sổ luôn thỏa "mọi ký tự `< k` lần" **trước** khi thêm `s[r]`. Thêm `s[r]` chỉ tăng đúng một số đếm, nên nếu bất biến bị phá thì kẻ phá chỉ có thể là `s[r]`. Khi co cửa sổ, các số đếm chỉ giảm ⇒ không ký tự nào khác vượt ngưỡng. Nhờ vậy mỗi bước kiểm tra `O(1)` thay vì `O(26)`.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

`"abacb"`, `k = 2`:

| `r` | ký tự | xâu con hợp lệ kết thúc tại `r` | `l(r)` |
|---|---|---|---|
| 0 | a | — | 0 |
| 1 | b | — | 0 |
| 2 | a | `aba` | 1 |
| 3 | c | `abac` | 1 |
| 4 | b | `abacb`, `bacb` | 2 |

Tổng `0 + 0 + 1 + 1 + 2 = 4` ✓. Ở `r = 4`, đầu trái nhỏ nhất **không** hợp lệ là `l = 2` (`"acb"`: mọi ký tự 1 lần).

## 5. Cách 1 — Brute Force: mọi xâu con

In [4]:
def brute(s, k):
    n = len(s)
    return sum(1 for l in range(n) for r in range(l, n) if max(Counter(s[l:r + 1]).values()) >= k)

for s, k, exp in examples:
    print(brute(s, k), '| kỳ vọng', exp)

4 | kỳ vọng 4
15 | kỳ vọng 15


**Độ phức tạp:** `O(n³)`.

## 6. Ý tưởng thuật toán / bước đột phá

Giữ cửa sổ `[l, r]` sao cho **mọi** ký tự trong cửa sổ xuất hiện `< k` lần, và `l` nhỏ nhất có thể. Khi thêm `c = s[r]`:

```
cnt[c] += 1
while cnt[c] >= k:          # chỉ c mới có thể vừa chạm ngưỡng
    cnt[s[l]] -= 1; l += 1
ans += l                    # các đầu trái 0..l-1 đều hợp lệ
```

Vì trước khi thêm `c` mọi số đếm `< k`, sau khi thêm chỉ `cnt[c]` có thể `≥ k` ⇒ chỉ cần kiểm tra `c`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — `O(n²)` với đếm tăng dần theo đầu trái (đủ cho bản I)

In [5]:
def quadratic(s, k):
    n, total = len(s), 0
    for l in range(n):
        cnt = [0] * 26
        for r in range(l, n):
            c = ord(s[r]) - 97
            cnt[c] += 1
            if cnt[c] >= k:
                total += n - r          # mọi r' >= r cũng hợp lệ
                break
    return total

for s, k, exp in examples:
    print(quadratic(s, k), '| kỳ vọng', exp)

4 | kỳ vọng 4
15 | kỳ vọng 15


Mẹo `break` + cộng `n − r` dùng tính đơn điệu theo **đầu phải**.

### 7.2. Cách 3 — Cửa sổ trượt (lời giải chính)

In [6]:
def sliding(s, k):
    cnt = [0] * 26
    ans = l = 0
    for ch in s:
        c = ord(ch) - 97
        cnt[c] += 1
        while cnt[c] >= k:
            cnt[ord(s[l]) - 97] -= 1
            l += 1
        ans += l
    return ans

for s, k, exp in examples:
    print(sliding(s, k), '| kỳ vọng', exp)

4 | kỳ vọng 4
15 | kỳ vọng 15


### 7.3. Cách 4 — Vị trí lần xuất hiện thứ `k` từ cuối

Với mỗi ký tự `c`, gọi `pos_c` là danh sách vị trí của `c` trong `s[0..r]`. `s[l..r]` chứa `c` ít nhất `k` lần ⇔ `l ≤ pos_c[−k]`. Vậy `l(r) = max_c (pos_c[−k] + 1)` (bỏ qua ký tự có `< k` lần). Chỉ ký tự `s[r]` thay đổi khi sang `r` mới ⇒ giữ `best = max` tăng dần.

In [7]:
def kth_last(s, k):
    pos = [[] for _ in range(26)]
    best = ans = 0
    for r, ch in enumerate(s):
        c = ord(ch) - 97
        pos[c].append(r)
        if len(pos[c]) >= k:
            best = max(best, pos[c][-k] + 1)
        ans += best
    return ans

for s, k, exp in examples:
    print(kth_last(s, k), '| kỳ vọng', exp)

4 | kỳ vọng 4
15 | kỳ vọng 15


`best` không giảm vì `pos_c[−k]` của mỗi ký tự chỉ tăng khi thêm phần tử.

## 8. TRACE từng bước

Trace cách 3 trên `"aabbaab"`, `k = 3`.

In [8]:
def trace(s, k):
    cnt = Counter(); ans = l = 0
    for r, c in enumerate(s):
        cnt[c] += 1
        moved = []
        while cnt[c] >= k:
            cnt[s[l]] -= 1; moved.append(s[l]); l += 1
        ans += l
        print(f"r = {r} ({c}): cửa sổ {s[l:r + 1]!r:9s} bỏ {''.join(moved) or '-':3s} l = {l} -> cộng {l}, ans = {ans}")
    print('đáp án:', ans, '| vét cạn:', brute(s, k))

trace("aabbaab", 3)

r = 0 (a): cửa sổ 'a'       bỏ -   l = 0 -> cộng 0, ans = 0
r = 1 (a): cửa sổ 'aa'      bỏ -   l = 0 -> cộng 0, ans = 0
r = 2 (b): cửa sổ 'aab'     bỏ -   l = 0 -> cộng 0, ans = 0
r = 3 (b): cửa sổ 'aabb'    bỏ -   l = 0 -> cộng 0, ans = 0
r = 4 (a): cửa sổ 'abba'    bỏ a   l = 1 -> cộng 1, ans = 1
r = 5 (a): cửa sổ 'bbaa'    bỏ a   l = 2 -> cộng 2, ans = 3
r = 6 (b): cửa sổ 'baab'    bỏ b   l = 3 -> cộng 3, ans = 6
đáp án: 6 | vét cạn: 6


## 9. Code nộp LeetCode

(Chữ ký theo doocs — xem ghi chú Premium.)

In [9]:
class Solution:
    def numberOfSubstrings(self, s: str, k: int) -> int:
        cnt = [0] * 26
        b = s.encode()
        ans = l = 0
        for x in b:
            c = x - 97
            cnt[c] += 1
            while cnt[c] >= k:
                cnt[b[l] - 97] -= 1
                l += 1
            ans += l
        return ans

for s, k, exp in examples:
    got = Solution().numberOfSubstrings(s, k)
    print(got, 'OK' if got == exp else 'SAI')

4 OK
15 OK


`s.encode()` cho mảng byte — truy cập `b[l]` nhanh hơn `ord(s[l])` một chút.

## 10. Vì sao thuật toán đúng

**Bất biến:** sau xử lý `r`, cửa sổ `[l, r]` là đoạn kết thúc tại `r` có **mọi** ký tự xuất hiện `< k` lần và `l` nhỏ nhất với tính chất đó.

- Khởi đầu đúng (cửa sổ rỗng). Thêm `s[r]`: chỉ `cnt[s[r]]` có thể đạt `k`; tiến `l` cho tới khi `cnt[s[r]] < k` — mọi `l` bị bỏ qua đều cho đoạn chứa `s[r]` ít nhất `k` lần (không thỏa bất biến), và `l` mới thỏa (mọi số đếm `< k`). Vì `l(r)` không giảm (mục 3.3) nên không cần lùi `l`.
- Khi đó `s[l'..r]` hợp lệ (có ký tự `≥ k` lần) ⇔ `l' < l` ⇒ có đúng `l` đầu trái hợp lệ.

Tổng theo `r` cho đáp án.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(n³)` | `O(n)` |
| 2. `O(n²)` + break | `O(n²)` | `O(26)` |
| 3. Cửa sổ trượt | `O(n)` | `O(26)` |
| 4. Vị trí thứ k từ cuối | `O(n)` | `O(n)` |

Đáp án tối đa `n(n+1)/2 ≈ 4.5·10^{10}` — vượt `int32`, trong C++/Java dùng `long long` / `long`.

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [10]:
impls = {'quadratic': quadratic, 'sliding': sliding, 'kth_last': kth_last,
         'submit': lambda s, k: Solution().numberOfSubstrings(s, k)}
edge = [("a", 1, 1), ("aaa", 1, 6), ("aaa", 3, 1), ("aaa", 4, 0), ("ab", 2, 0), ("abab", 2, 3), ("zzzzz", 2, 10)]
for s, k, exp in examples + edge:
    assert brute(s, k) == exp, (s, k, brute(s, k))
    for name, f in impls.items():
        assert f(s, k) == exp, (name, s, k)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn

In [11]:
import random
rng = random.Random(3329)
cnt = 0
for _ in range(800):
    s = ''.join(rng.choice('abc'[:rng.randint(1, 3)]) for _ in range(rng.randint(1, 12)))
    k = rng.randint(1, len(s))
    exp = brute(s, k)
    for name, f in impls.items():
        assert f(s, k) == exp, (name, s, k)
    cnt += 1
print('Khớp vét cạn trên', cnt, 'bộ test.')

Khớp vét cạn trên 800 bộ test.


### 12.3. Ca lớn `n = 3·10^5`

In [12]:
import time
N = 3 * 10 ** 5
big = {
    'ngẫu nhiên 26 chữ, k = 3': (''.join(rng.choice('abcdefghijklmnopqrstuvwxyz') for _ in range(N)), 3),
    'toàn a, k = 1': ('a' * N, 1),
    'toàn a, k = N': ('a' * N, N),
    'xoay vòng 26 chữ, k = 1000': (''.join(chr(97 + i % 26) for i in range(N)), 1000),
}
for name, (s, k) in big.items():
    t = time.time(); a = Solution().numberOfSubstrings(s, k); t1 = time.time() - t
    t = time.time(); b = kth_last(s, k); t2 = time.time() - t
    assert a == b
    print(f"{name:28s}: {a:12d} | nộp {t1:.2f}s | cách 4 {t2:.2f}s")

ngẫu nhiên 26 chữ, k = 3    :  44995337001 | nộp 0.04s | cách 4 0.07s
toàn a, k = 1               :  45000150000 | nộp 0.04s | cách 4 0.07s
toàn a, k = N               :            1 | nộp 0.03s | cách 4 0.03s


xoay vòng 26 chữ, k = 1000  :  37545261351 | nộp 0.04s | cách 4 0.08s


Kiểm tra độc lập: `"a"*N` với `k` bất kỳ ⇒ xâu con độ dài `≥ k`: `Σ_{len=k}^{N} (N − len + 1) = (N − k + 1)(N − k + 2)/2`.

In [13]:
for k in [1, 2, 1000, N]:
    assert Solution().numberOfSubstrings('a' * N, k) == (N - k + 1) * (N - k + 2) // 2
print('toàn một chữ cái: khớp công thức.')

toàn một chữ cái: khớp công thức.


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng `>` thay vì `>=` trong điều kiện co cửa sổ

In [14]:
def wrong_strict(s, k):
    cnt = Counter(); ans = l = 0
    for c in s:
        cnt[c] += 1
        while cnt[c] > k:                   # SAI: thực chất đếm "ít nhất k+1 lần"
            cnt[s[l]] -= 1; l += 1
        ans += l
    return ans

print('sai :', wrong_strict("abacb", 2), '| đúng:', Solution().numberOfSubstrings("abacb", 2))

sai : 0 | đúng: 4


**Vì sao sai:** cửa sổ phải giữ mọi số đếm **`< k`**; dùng `> k` cho phép số đếm bằng `k` trong cửa sổ ⇒ bỏ sót các xâu con có ký tự xuất hiện đúng `k` lần.

### Lỗi 2 — Cộng độ dài cửa sổ `r − l + 1` thay vì `l`

In [15]:
def wrong_window_len(s, k):
    cnt = Counter(); ans = l = 0
    for r, c in enumerate(s):
        cnt[c] += 1
        while cnt[c] >= k:
            cnt[s[l]] -= 1; l += 1
        ans += r - l + 1                    # SAI: đây là số xâu con KHÔNG hợp lệ
    return ans

print('sai :', wrong_window_len("abacb", 2), '| đúng:', Solution().numberOfSubstrings("abacb", 2))

sai : 11 | đúng: 4


**Vì sao sai:** cửa sổ chứa các đầu trái **không** hợp lệ. Nếu muốn dùng nó thì phải lấy phần bù: `n(n+1)/2 − Σ (r − l + 1)` (`15 − 11 = 4`).

### Lỗi 3 — Đếm xâu con **phân biệt** (dùng `set`)

In [16]:
def wrong_distinct(s, k):
    n = len(s)
    return len({s[l:r + 1] for l in range(n) for r in range(l, n) if max(Counter(s[l:r + 1]).values()) >= k})

print('sai :', wrong_distinct("aaa", 1), '| đúng:', Solution().numberOfSubstrings("aaa", 1))

sai : 3 | đúng: 6


**Vì sao sai:** đề đếm theo **vị trí** (ví dụ 2: `"abcde"` có 15 xâu con = `5·6/2`); `"aaa"` có 6 xâu con theo vị trí nhưng chỉ 3 xâu khác nhau.

### Lỗi 4 — Chỉ co cửa sổ một bước (`if` thay vì `while`)

In [17]:
def wrong_if(s, k):
    cnt = Counter(); ans = l = 0
    for c in s:
        cnt[c] += 1
        if cnt[c] >= k:                     # SAI: có thể phải bỏ nhiều ký tự
            cnt[s[l]] -= 1; l += 1
        ans += l
    return ans

bad = "baa"
print('sai :', wrong_if(bad, 2), '| đúng:', Solution().numberOfSubstrings(bad, 2))

sai : 1 | đúng: 2


**Vì sao sai:** ở `r = 2`, `cnt[a] = 2`; bỏ `s[0] = 'b'` chưa đủ (cửa sổ `"aa"` vẫn có `a` hai lần) — phải bỏ tiếp `s[1]`. Đáp án đúng: `"baa"`, `"aa"` ⇒ 2.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đếm xâu con thỏa tính chất **đơn điệu khi mở rộng**" ⇒ hai con trỏ, mỗi `r` cộng số đầu trái hợp lệ.
- Tính chất "có ký tự ≥ k lần" khó duy trì trực tiếp, nhưng **phần bù** "mọi ký tự < k lần" dễ duy trì trong cửa sổ.
- Khi chỉ một phần tử thay đổi mỗi bước, chỉ cần kiểm tra phần tử đó.

## 15. Biến thể của bài

1. **LC 3325 (bản I):** `n ≤ 3000`.
2. **Mọi** ký tự xuất hiện `≥ k` lần (LC 395 họ "Longest Substring with At Least K Repeating") — không đơn điệu, cần chia để trị hoặc cố định số ký tự phân biệt.
3. Có ít nhất **hai** ký tự khác nhau xuất hiện `≥ k` lần ⇒ `l(r)` = giá trị lớn thứ hai trong `{pos_c[−k] + 1}`.
4. Đếm xâu con có ký tự xuất hiện **đúng** `k` lần (không đơn điệu — dùng vị trí).

In [18]:
# Biến thể 3: ít nhất hai ký tự khác nhau xuất hiện >= k lần
def two_chars(s, k):
    pos = [[] for _ in range(26)]
    vals = [0] * 26
    ans = 0
    for r, ch in enumerate(s):
        c = ord(ch) - 97
        pos[c].append(r)
        if len(pos[c]) >= k:
            vals[c] = pos[c][-k] + 1
        top = sorted(vals, reverse=True)
        ans += top[1]
    return ans

def two_chars_brute(s, k):
    n = len(s)
    return sum(1 for l in range(n) for r in range(l, n)
               if sum(1 for v in Counter(s[l:r + 1]).values() if v >= k) >= 2)

rng = random.Random(15)
for _ in range(300):
    s = ''.join(rng.choice('abc') for _ in range(rng.randint(1, 10)))
    k = rng.randint(1, 4)
    assert two_chars(s, k) == two_chars_brute(s, k)
print('Biến thể "hai ký tự >= k lần" khớp vét cạn.')

Biến thể "hai ký tự >= k lần" khớp vét cạn.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Mọi xâu con | `O(n³)` | Oracle |
| 2 | Theo đầu trái + break | `O(n²)` | Bản I |
| 3 | Cửa sổ trượt | `O(n)` | **Nộp bài** |
| 4 | Vị trí thứ k từ cuối | `O(n)` | Mở rộng được (biến thể 3) |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `s = "aaaa"`, `k = 2`.

**Bài 2 (tính tay):** `s = "abba"`, `k = 2`.

**Bài 3 (điền TODO):** đếm xâu con mà **mọi** ký tự xuất hiện `< k` lần (phần bù).

**Bài 4 (tự viết lại):** độ dài **ngắn nhất** của xâu con hợp lệ (hoặc `-1`).

**Bài 5 (mini challenge):** biến thể 4 — có ký tự xuất hiện **đúng** `k` lần.

In [19]:
def complement_todo(s, k):
    # TODO
    raise NotImplementedError

def shortest_valid_todo(s, k):
    # TODO
    raise NotImplementedError

def exactly_k_todo(s, k):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Xâu con độ dài `≥ 2`. (2) `3 + 2 + 1`. (3) `6`.

**Bài 2.** (1) Trace `l(r)`: `0, 0, 2, 2`. (2) `r = 2`: `"abb"`, `"bb"`; `r = 3`: `"abba"`, `"bba"`. (3) `4`.

**Bài 3.** (1) `Σ (r − l + 1)`. (2) Hoặc `n(n+1)/2 − đáp án`. (3) Kiểm tra cả hai.

**Bài 4.** (1) Xâu con ngắn nhất hợp lệ phải kết thúc ở lần xuất hiện thứ `k` của ký tự nào đó và bắt đầu ở lần xuất hiện đầu tiên trong `k` lần ấy. (2) `min_c min_i (pos_c[i + k − 1] − pos_c[i] + 1)`. (3) Vét cạn.

**Bài 5.** (1) Với `r` cố định, ký tự `c` xuất hiện đúng `k` lần trong `s[l..r]` ⇔ `l ∈ (pos_c[−k−1], pos_c[−k]]`. (2) Hợp các khoảng theo `c` (≤ 26 khoảng). (3) Đếm độ dài hợp.

## 19. Đáp án tham khảo cho bài tập

In [20]:
# Bài 1, 2, 3
assert Solution().numberOfSubstrings("aaaa", 2) == 6
assert Solution().numberOfSubstrings("abba", 2) == 4 == brute("abba", 2)

def complement(s, k):
    cnt = [0] * 26; ans = l = 0
    for r, ch in enumerate(s):
        c = ord(ch) - 97
        cnt[c] += 1
        while cnt[c] >= k:
            cnt[ord(s[l]) - 97] -= 1; l += 1
        ans += r - l + 1
    return ans

rng = random.Random(3)
for _ in range(300):
    s = ''.join(rng.choice('abc') for _ in range(rng.randint(1, 10))); k = rng.randint(1, 4)
    n = len(s)
    assert complement(s, k) == n * (n + 1) // 2 - Solution().numberOfSubstrings(s, k)
print('Bài 1: 6 | Bài 2: 4 | Bài 3: phần bù khớp.')

Bài 1: 6 | Bài 2: 4 | Bài 3: phần bù khớp.


In [21]:
# Bài 4, 5
def shortest_valid(s, k):
    pos = {}
    for i, c in enumerate(s):
        pos.setdefault(c, []).append(i)
    best = min((p[i + k - 1] - p[i] + 1 for p in pos.values() for i in range(len(p) - k + 1)), default=-1)
    return best

def exactly_k(s, k):
    pos = [[] for _ in range(26)]
    ans = 0
    for r, ch in enumerate(s):
        pos[ord(ch) - 97].append(r)
        intervals = []
        for p in pos:
            if len(p) >= k:
                lo = p[-k - 1] + 1 if len(p) > k else 0
                intervals.append((lo, p[-k]))          # l ∈ [lo, p[-k]]
        intervals.sort()
        covered, cur_end = 0, -1
        for a, b in intervals:
            if b > cur_end:
                covered += b - max(a, cur_end + 1) + 1
                cur_end = b
        ans += covered
    return ans

rng = random.Random(4)
for _ in range(300):
    s = ''.join(rng.choice('abc') for _ in range(rng.randint(1, 10))); k = rng.randint(1, 4)
    n = len(s)
    lens = [r - l + 1 for l in range(n) for r in range(l, n) if max(Counter(s[l:r + 1]).values()) >= k]
    assert shortest_valid(s, k) == (min(lens) if lens else -1)
    assert exactly_k(s, k) == sum(1 for l in range(n) for r in range(l, n) if k in Counter(s[l:r + 1]).values())
print('Bài 4, 5 khớp vét cạn.')

Bài 4, 5 khớp vét cạn.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
cnt = [0]*26; ans = l = 0
for ch in s:
    cnt[ch] += 1
    while cnt[ch] >= k: cnt[s[l]] -= 1; l += 1
    ans += l
return ans
```

**Checklist:**

- [ ] Điều kiện co là `>= k`?
- [ ] `while`, không phải `if`?
- [ ] Cộng `l` (số đầu trái hợp lệ), không phải độ dài cửa sổ?
- [ ] Đếm theo vị trí (không khử trùng lặp)?
- [ ] Kiểu dữ liệu 64-bit ở ngôn ngữ khác?